## Backend

In [2]:
import pandas as pd
import numpy as np
import random

In [3]:
class DistUtils():

  @staticmethod
  def bounded_normal(min_val, max_val, size=1000, std_ratio=0.25, clip=True):
    """
    Generate samples from a normal distribution bounded by [min_val, max_val].

    - std_ratio: fraction of the range used for std (0.25 → 4σ spans full range)
    - clip: if True, values are clipped to the range
    """
    mean = (max_val + min_val) / 2
    std = (max_val - min_val) * std_ratio
    samples = np.random.normal(mean, std, size)

    if clip:
      samples = np.clip(samples, min_val, max_val)

    return samples

  @staticmethod
  def bounded_exponential(min_val, max_val, size=1000, scale_ratio=0.2, reverse=False):
    """
    Exponential distribution bounded within [min_val, max_val].

    - scale_ratio: controls steepness (smaller = steeper decay)
    - reverse: if True, decay starts from max side instead of min
    """
    if scale_ratio < 0:
      scale_ratio = -scale_ratio
      reverse = not reverse
    scale = (max_val - min_val) * scale_ratio

    samples = np.random.exponential(scale=scale, size=size)
    samples = min_val + samples  # shift to start at min

    samples = np.clip(samples, min_val, max_val)

    if reverse:
      samples = max_val - (samples - min_val)

    return samples

  @staticmethod
  def bounded_lognormal(min_val, max_val, size=1000, sigma_ratio=0.25, clip=True):
    """
    Generate samples following a lognormal-like shape within [min_val, max_val].

    - sigma_ratio: controls spread relative to range (higher = more skew)
    - clip: whether to clip values to [min_val, max_val]
    """
    # Compute parameters of the underlying normal
    mean = 0
    if(sigma_ratio < 0): sigma = -sigma_ratio
    else: sigma = sigma_ratio

    # Generate raw lognormal
    raw = np.random.lognormal(mean, sigma, size=size)

    # Normalize to 0–1 range
    raw = (raw - raw.min()) / (raw.max() - raw.min())

    # Scale to [min_val, max_val]
    samples = min_val + raw * (max_val - min_val)

    if clip:
      samples = np.clip(samples, min_val, max_val)

    if(sigma_ratio < 0): samples = max_val - (samples - min_val)

    return samples



In [4]:
class JobSchedulingProblem():

  def __init__(self, job_machine_times: pd.DataFrame):
    self.jm_times = job_machine_times
    self.n_jobs = job_machine_times.shape[0]
    self.n_machines = job_machine_times.shape[1]
    self.job_names = job_machine_times.index
    self.machine_names = job_machine_times.columns

  def get_job_time(self, job, machine):
    return self.jm_times.loc[job, machine]

  def build_schedule(self, assignment):
    schedules = []
    for machine in assignment.unique():
      machine_assig = assignment[assignment == machine]
      machine_assig = machine_assig.reset_index()
      machine_assig.columns = ['job', 'machine']
      machine_assig['job_time'] = machine_assig["job"].apply(lambda job: self.get_job_time(job, machine))
      machine_assig['start_time'] = machine_assig['job_time'].cumsum().shift(1, fill_value=0)
      machine_assig['end_time'] = machine_assig['job_time'].cumsum()
      schedules.append(machine_assig)
    schedule = pd.concat(schedules).reset_index(drop=True)
    return schedule

  @staticmethod
  def create_random_jm_times(n_jobs=3, n_machines=2, min_time=1, max_time=5, distribution="uniform", noise=None, index=None, columns=None, **kwargs):

    """
    Method to build a random job-to-machine time matrix.
    Parameters
    ----------
    n_jobs : int
        Number of jobs.
    n_machines : int
        Number of machines.
    min_time : int
        Minimum time for a job to be scheduled on a machine.
    max_time : int
        Maximum time for a job to be scheduled on a machine.
    distribution : str
        Distribution of the time for a job to be scheduled on a machine.
        Can be "uniform", "constant", "normal", "exponential", "lognormal", "same_per_machine", "same_per_job".
    noise : float
        Value from 0 to 1 indicating how much zeros will randomly overwrite the matrix.
    index : list
        List of job names.
    columns : list
        List of machine names.
    Returns
    -------
    pd.DataFrame
        Random job-to-machine time matrix.
    """

    if index is None:
      index = [f"job_{i}" for i in range(n_jobs)]
    if columns is None:
      columns = [f"machine_{i}" for i in range(n_machines)]

    if distribution == "uniform":
      dist = np.random.randint(min_time, max_time, (n_jobs, n_machines))
    elif distribution == "constant":
      dist = np.ones((n_jobs, n_machines))*min_time
    elif distribution == "normal":
      dist = DistUtils.bounded_normal(min_time, max_time, size=(n_jobs, n_machines))
    elif distribution == "exponential":
      dist = DistUtils.bounded_exponential(min_time, max_time, size=(n_jobs, n_machines), **kwargs)
    elif distribution == "lognormal":
      dist = DistUtils.bounded_lognormal(min_time, max_time, size=(n_jobs, n_machines), **kwargs)
    elif distribution == "same_per_machine":
      times = np.random.randint(min_time, max_time, (n_machines, 1))
      dist = np.hstack([times for i in range(n_jobs)]).T
    elif distribution == "same_per_job":
      times = np.random.randint(min_time, max_time, (1, n_jobs))
      dist = np.vstack([times for i in range(n_machines)]).T
    else:
      raise NotImplementedError("Distribution not implemented")

    if noise is not None:
      orig_shape = dist.shape
      dist_flat = dist.flatten()
      n_zeros = int(noise * len(dist_flat))
      zeros = np.random.choice(len(dist_flat), n_zeros, replace=False)
      dist_flat[zeros] = 0
      dist = dist_flat.reshape(orig_shape)

    dist = dist.astype(int)

    return pd.DataFrame(dist, index=index, columns=columns)

  @staticmethod
  def create_random_jobshop_dataset(n_jobs=3, n_operations=3, machine_names=None, job_names=None, **kwargs):

    if machine_names is None: machine_names = [f"machine_{i}" for i in range(n_operations)]
    if job_names is None: job_names = [f"job_{i}" for i in range(n_jobs)]
    operation_names = [i for i in range(n_operations)]
    job_times = JobSchedulingProblem.create_random_jm_times(n_jobs, n_operations, columns=operation_names, index=job_names, **kwargs)
    job_machines = pd.DataFrame([
        np.random.choice(machine_names, n_operations, replace=False)
        for _ in range(n_jobs)]
        , index=job_times.index, columns=job_times.columns)
    return job_times, job_machines

  @staticmethod
  def compute_metrics(schedule):

    machine_load = schedule.groupby('machine')['end_time'].max()
    n_jobs_scheduled = len(schedule["job"].unique())
    n_machines_scheduled = len(schedule["machine"].unique())
    makespan = machine_load.max()
    flow_times = schedule.groupby(by='job')[["end_time", "start_time"]].apply(lambda x: x["end_time"].max() - x["start_time"].min())
    avg_flow_time = int(flow_times.mean())

    return {
        "makespan": makespan,
        "avg_flow_time": avg_flow_time,
        "n_jobs_scheduled": n_jobs_scheduled,
        "n_machines_scheduled": n_machines_scheduled,
        "n_operations_scheduled": len(schedule)
    }

class JobToMachineProblem(JobSchedulingProblem):

  def __init__(self, job_machine_times: pd.DataFrame, capacity):
    super().__init__(job_machine_times)
    self.capacity = capacity

class FlowShopProblem(JobSchedulingProblem):

  def build_schedule(self, order):

    jm_times = np.array(self.jm_times.loc[order])
    start = np.zeros((self.n_jobs, self.n_machines))
    end = np.zeros((self.n_jobs, self.n_machines))

    for i in range(jm_times.shape[0]):
      for m in range(self.n_machines):
        if i == 0 and m == 0:
          start[i, m] = 0
        elif i == 0:
          start[i, m] = end[i, m-1]
        elif m == 0:
          start[i, m] = end[i-1, m]
        else:
          start[i, m] = max(end[i-1, m], end[i, m-1])
        end[i, m] = start[i, m] + jm_times[i, m]

    # Build readable schedule
    schedule = []
    for i, job in enumerate(list(order)):
      for m, machine in enumerate(self.machine_names):
        schedule.append({
          "job": job,
          "machine": machine,
          "job_time": jm_times[i, m],
          "start_time": start[i, m],
          "end_time": end[i, m]
        })

    schedule = pd.DataFrame(schedule)
    schedule = schedule[schedule["job_time"] != 0]

    return schedule

class JobShopProblem(JobSchedulingProblem):

  def __init__(self, job_times: pd.DataFrame, job_machines: pd.DataFrame):
    self.job_times = job_times
    self.job_machines = job_machines
    self.n_jobs = job_times.shape[0]
    self.n_ops = job_times.shape[1]
    self.n_machines = job_machines.shape[1]
    self.job_names = job_times.index
    self.machine_names = list(set(job_machines.values.flatten()))
    self.operation_names = job_machines.columns
    self.original_order = job_machines.index

  def restore_order(self):
    self.job_machines = self.job_machines.loc[self.original_order]
    self.job_times = self.job_times.loc[self.original_order]

  def shuffle_jobs(self):
    self.job_machines = self.job_machines.sample(frac=1)
    self.job_times = self.job_times.loc[self.job_machines.index]

  @staticmethod
  def verify_operation_order(schedule):
    orders = list(
      schedule
      .sort_values(by=["job", "operation"])
      .groupby(by="job")[["start_time", "end_time"]]
      .apply(lambda x: np.array(x.values).flatten())
      .values
    )
    operation_order_respected = all([all(i == sorted(i)) for i in orders])
    return operation_order_respected

  @staticmethod
  def verify_machines_non_parallel(schedule):
    orders = list(
      schedule
      .sort_values(by=["machine", "start_time"])
      .groupby(by="machine")[["start_time", "end_time"]]
      .apply(lambda x: np.array(x.values).flatten())
      .values
    )
    machine_non_parallel = all([all(i == sorted(i)) for i in orders])
    return machine_non_parallel

  @staticmethod
  def verify_schedule(schedule):
    return JobShopProblem.verify_operation_order(schedule) and JobShopProblem.verify_machines_non_parallel(schedule)

  def build_schedule(self, operation_order: pd.DataFrame, verify=False, drop_zeros=True):

    machine_map = {j:i for i, j in enumerate(self.machine_names)}
    job_map = {j:i for i, j in enumerate(self.job_names)}
    operation_map = {j:i for i, j in enumerate(self.operation_names)}

    operation_order["job"] = operation_order["job"].map(job_map)
    operation_order["machine"] = operation_order["machine"].map(machine_map)
    operation_order["operation"] = operation_order["operation"].map(operation_map)

    # Propagate times in terms of operation order
    times = np.array(self.job_times)
    start_time = np.zeros((self.n_jobs, self.n_ops))
    end_time = np.zeros((self.n_jobs, self.n_ops))
    machine_end_times = np.zeros(self.n_machines)
    for i, row in operation_order.iterrows():
      op, job, m = row["operation"], row["job"], row["machine"]
      machine_end_time = machine_end_times[m]
      job_end_time = end_time[job].max()
      start_time[job][op] = max(machine_end_time, job_end_time)
      end_time[job][op] = start_time[job][op] + times[job, op]
      machine_end_times[m] = end_time[job][op]

    # Finalizing schedule
    schedule = operation_order.sort_values(by=["job", "operation"])
    schedule["job"] = schedule["job"].apply(lambda x: self.job_names[x])
    schedule["machine"] = schedule["machine"].apply(lambda x: self.machine_names[x])
    schedule["operation"] = schedule["operation"].apply(lambda x: self.operation_names[x])
    schedule["job_time"] = times.flatten()
    schedule["start_time"] = start_time.flatten().astype(int)
    schedule["end_time"] = end_time.flatten().astype(int)

    if drop_zeros: schedule = schedule[schedule["job_time"] != 0]

    if not self.verify_schedule(schedule[schedule["job_time"] != 0]):
      if verify: raise Exception("Cannot schedule, operations have conflicts")
      else: print("Warning: This schedule has timing conflicts")

    return schedule


In [5]:
# JobToMachine solvers

class MLFSolver():

  """
  Greedy solver for Job-to-Machine assignment using Min Load First heuristic.
  """

  @staticmethod
  def solve(problem: JobToMachineProblem):
    assignment = pd.Series([None]*problem.n_jobs, index=problem.job_names).rename("assignment")
    machine_load = pd.Series(np.zeros(problem.n_machines), index=problem.machine_names)
    for _, job in problem.jm_times.iterrows():
      for machine, time in job.sort_values().items():
        load = machine_load[machine]
        if load + time <= problem.capacity:
          machine_load[machine] += time
          assignment[job.name] = machine
          break
    return problem.build_schedule(assignment)

class SJFSolver():

  """
  Greedy solver for Job-to-Machine assignment using Shortest Job First heuristic.
  """

  @staticmethod
  def solve(problem: JobToMachineProblem):
    problem.jm_times = problem.jm_times.loc[problem.jm_times.min(axis=1).sort_values().index]
    return MLFSolver.solve(problem)

class LJFSolver():

  """
  Greedy solver for Job-to-Machine assignment using Longest Job First heuristic.
  """

  @staticmethod
  def solve(problem: JobToMachineProblem):
    problem.jm_times = problem.jm_times.loc[problem.jm_times.max(axis=1).sort_values().index]
    return MLFSolver.solve(problem)



In [6]:
# FlowShop solvers

class FSDefaultOrderSolver():

  """
  Solver that creates a schedule using same order of original jobs.
  """

  @staticmethod
  def solve(problem: FlowShopProblem):
    order = problem.jm_times.index
    return problem.build_schedule(order)

class FSRandomSolver():

  """
  Solver that creates a schedule using quasi-brute force random sampling.
  """
  @staticmethod
  def solve(problem: FlowShopProblem, max_trys=10):
    best_schedule = None
    best_makespan = np.inf
    for _ in range(int(max_trys)):
      order = list(problem.job_names)
      random.shuffle(order)
      schedule = problem.build_schedule(order)
      makespan = JobSchedulingProblem.compute_metrics(schedule)["makespan"]
      if makespan < best_makespan:
        best_makespan = makespan
        best_schedule = schedule
    return best_schedule

class NEHSolver():
  """
  Solver that creates a schedule using Nawaz-Enscore-Ham heuristic.
  """
  @staticmethod
  def compute_makespan(seq, times):
    n, m = len(seq), times.shape[1]
    completion = np.zeros((n, m))
    completion[0, 0] = times[seq[0], 0]

    # First row and first column
    for j in range(1, m):
      completion[0, j] = completion[0, j-1] + times[seq[0], j]
    for i in range(1, n):
      completion[i, 0] = completion[i-1, 0] + times[seq[i], 0]

    # Fill the rest
    for i in range(1, n):
      for j in range(1, m):
        completion[i, j] = max(completion[i-1, j], completion[i, j-1]) + times[seq[i], j]
    return completion[-1, -1]

  @staticmethod
  def __solve(times: np.ndarray):
    # times: jobs x machines matrix
    job_sums = times.sum(axis=1)
    jobs_sorted = np.argsort(-job_sums)  # descending
    order = []

    for job in jobs_sorted:
      best_seq, best_makespan = None, float('inf')
      for pos in range(len(order) + 1):
        new_seq = order[:pos] + [job] + order[pos:]
        makespan = NEHSolver.compute_makespan(new_seq, times)
        if makespan < best_makespan:
          best_seq, best_makespan = new_seq, makespan
      order = best_seq
    return order, best_makespan

  @staticmethod
  def solve(problem: FlowShopProblem):
    order, _  = NEHSolver.__solve(problem.jm_times.to_numpy())
    order = problem.jm_times.index[order]
    return problem.build_schedule(order)


In [7]:
# JobShop solvers

class JSDefaultOrderSolver():

  """
  Solver that creates a schedule using same order of original jobs.
  Option to shuffle.
  """

  @staticmethod
  def solve(problem: JobShopProblem, shuffle=False, verify=False, drop_zeros=True):
    base_schedule = []
    for op in problem.job_machines.columns:
      base_schedule += [[j, m, op] for j, m in problem.job_machines[op].items()]
    base_schedule = pd.DataFrame(base_schedule, columns=["job", "machine", "operation"])
    if shuffle: base_schedule = base_schedule.sample(frac=1)
    return problem.build_schedule(base_schedule, verify=verify, drop_zeros=drop_zeros)

class JSRandomSolver():

  """
  Solver that creates a schedule using quasi-brute force random sampling.
  """


  @staticmethod
  def solve(problem: JobShopProblem, max_trys=10, shuffle_type="job"):
    """
    if shuffle type == "job", shuffle whole jobs (operation packages)
    if shuffle type == "operation", shuffle at indidividual operation level.
    """
    best_schedule = JSDefaultOrderSolver.solve(problem)
    best_makespan = JobSchedulingProblem.compute_metrics(best_schedule)["makespan"]
    for _ in range(int(max_trys)):
      try:
        if shuffle_type == "job": problem.shuffle_jobs()
        schedule = JSDefaultOrderSolver.solve(problem, shuffle=(shuffle_type=="operation"), verify=True)
      except: continue
      makespan = JobSchedulingProblem.compute_metrics(schedule)["makespan"]
      if makespan < best_makespan:
        best_makespan = makespan
        best_schedule = schedule

    problem.restore_order()
    return best_schedule


class SPTSolver():
  """
  Solver that creates a schedule using Shortest Processing Time heuristic.
  """
  @staticmethod
  def solve(problem: JobShopProblem, step_size=1):
    base_schedule = JSDefaultOrderSolver.solve(problem,drop_zeros=False)
    new_schedule = []
    next_ops = {i:0 for i in problem.job_names}
    machine_loads = {i:0 for i in problem.machine_names}
    curr_step = 0
    while len(base_schedule) != 0:
      candidate_machines = [k for k,v in machine_loads.items() if v <= curr_step]
      for machine in candidate_machines:
        while machine_loads[machine] <= curr_step:
          # Find the operations involving machine
          candidate_ops = base_schedule[base_schedule["machine"] == machine].reset_index()
          # Assign the one with least operation time
          candidate_ops.index = candidate_ops["job"]+"_"+candidate_ops["operation"].astype(str)
          target_idx = [f"{k}_{v}" for k,v in next_ops.items()]
          candidate_ops = candidate_ops.loc[candidate_ops.index.intersection(target_idx)]
          if(candidate_ops.empty): break
          # Assign
          next_job = candidate_ops.sort_values(by="job_time").iloc[0]
          next_ops[next_job["job"]] += 1
          machine_loads[next_job["machine"]] += next_job["job_time"]
          new_schedule.append([next_job["job"], next_job["machine"], next_job["operation"]])
          base_schedule = base_schedule.drop(next_job["index"])
      curr_step += step_size
    new_schedule = pd.DataFrame(new_schedule, columns=["job", "machine", "operation"])
    return problem.build_schedule(new_schedule)


In [8]:
jm_times = JobSchedulingProblem.create_random_jm_times(n_jobs=200, n_machines=5, min_time=1, max_time=100, distribution="uniform")
capacity = int(jm_times.quantile(q=0.1, axis=1).sum()/5)
print("Capacity:", capacity)
problem = JobToMachineProblem(jm_times, capacity=capacity)
print(JobSchedulingProblem.compute_metrics(MLFSolver.solve(problem)))
print(JobSchedulingProblem.compute_metrics(SJFSolver.solve(problem)))
print(JobSchedulingProblem.compute_metrics(LJFSolver.solve(problem)))

Capacity: 978
{'makespan': 818, 'avg_flow_time': 17, 'n_jobs_scheduled': 200, 'n_machines_scheduled': 5, 'n_operations_scheduled': 200}
{'makespan': 818, 'avg_flow_time': 17, 'n_jobs_scheduled': 200, 'n_machines_scheduled': 5, 'n_operations_scheduled': 200}
{'makespan': 818, 'avg_flow_time': 17, 'n_jobs_scheduled': 200, 'n_machines_scheduled': 5, 'n_operations_scheduled': 200}


In [9]:
jm_times = JobSchedulingProblem.create_random_jm_times(n_jobs=200, n_machines=5, min_time=1, max_time=100, distribution="uniform")
problem = FlowShopProblem(jm_times)
print(JobSchedulingProblem.compute_metrics(FSDefaultOrderSolver.solve(problem)))
print(JobSchedulingProblem.compute_metrics(FSRandomSolver.solve(problem, max_trys=50)))
print(JobSchedulingProblem.compute_metrics(NEHSolver.solve(problem)))

{'makespan': 10717.0, 'avg_flow_time': 760, 'n_jobs_scheduled': 200, 'n_machines_scheduled': 5, 'n_operations_scheduled': 1000}
{'makespan': 11041.0, 'avg_flow_time': 994, 'n_jobs_scheduled': 200, 'n_machines_scheduled': 5, 'n_operations_scheduled': 1000}
{'makespan': 10569.0, 'avg_flow_time': 908, 'n_jobs_scheduled': 200, 'n_machines_scheduled': 5, 'n_operations_scheduled': 1000}


In [10]:
job_times, job_machines = JobSchedulingProblem.create_random_jobshop_dataset(n_jobs=100, n_operations=10, min_time=1, max_time=100, distribution="uniform")
problem = JobShopProblem(job_times, job_machines)
print(JobSchedulingProblem.compute_metrics(JSDefaultOrderSolver.solve(problem)))
print(JobSchedulingProblem.compute_metrics(JSRandomSolver.solve(problem, max_trys=20)))
print(JobSchedulingProblem.compute_metrics(SPTSolver.solve(problem)))

{'makespan': 6625, 'avg_flow_time': 5578, 'n_jobs_scheduled': 100, 'n_machines_scheduled': 10, 'n_operations_scheduled': 1000}
{'makespan': 5735, 'avg_flow_time': 4961, 'n_jobs_scheduled': 100, 'n_machines_scheduled': 10, 'n_operations_scheduled': 1000}
{'makespan': 11879, 'avg_flow_time': 4955, 'n_jobs_scheduled': 100, 'n_machines_scheduled': 10, 'n_operations_scheduled': 1000}


In [11]:
job_machines

,0,1,2,3,4,5,6,7,8,9
job_0,machine_5,machine_9,machine_2,machine_4,machine_7,machine_8,machine_3,machine_1,machine_0,machine_6
job_1,machine_2,machine_0,machine_7,machine_1,machine_6,machine_8,machine_3,machine_9,machine_5,machine_4
job_2,machine_6,machine_1,machine_7,machine_3,machine_0,machine_8,machine_4,machine_5,machine_2,machine_9
job_3,machine_4,machine_0,machine_3,machine_5,machine_8,machine_7,machine_1,machine_9,machine_6,machine_2
job_4,machine_5,machine_0,machine_4,machine_6,machine_9,machine_7,machine_3,machine_8,machine_1,machine_2
...,...,...,...,...,...,...,...,...,...,...
job_95,machine_3,machine_8,machine_5,machine_0,machine_6,machine_7,machine_4,machine_1,machine_9,machine_2
job_96,machine_3,machine_2,machine_4,machine_1,machine_9,machine_0,machine_6,machine_5,machine_7,machine_8
job_97,machine_4,machine_0,machine_7,machine_8,machine_2,machine_9,machine_3,machine_1,machine_6,machine_5
job_98,machine_8,machine_4,machine_5,machine_9,machine_0,machine_6,machine_1,machine_2,machine_3,machine_7


In [12]:
job_times

,0,1,2,3,4,5,6,7,8,9
job_0,61,2,41,15,10,60,48,56,65,71
job_1,8,68,82,16,51,67,48,29,85,71
job_2,72,28,26,84,89,10,44,62,43,19
job_3,65,88,51,21,29,59,89,68,41,49
job_4,43,61,23,98,73,23,67,34,7,76
...,...,...,...,...,...,...,...,...,...,...
job_95,63,17,29,20,4,22,11,57,44,97
job_96,15,89,10,66,30,70,22,70,11,61
job_97,61,61,58,57,51,33,4,90,24,28
job_98,70,37,78,47,15,81,40,80,88,54


## Testing & Benchmarking

In [13]:
from faker import Faker
import random

ALL_DISTRIBUTIONS = [
    "uniform",
    "constant",
    "normal",
    "exponential",
    "lognormal",
    "same_per_machine",
    "same_per_job"
]

fake = Faker('es_CO')
en_fake = Faker('en_US')
NOMBRES = list(set([" ".join(fake.name().split()[:2]) for _ in range(300)]))
NOMBRES_MASCULINOS = list(set([fake.first_name_male() for _ in range(20)]))
DIRECCIONES = list(set([fake.address().replace("\n", " ") for _ in range(20)]))

def generar_telefono_do():
    prefijos = ['809', '829', '849']
    return f"{random.choice(prefijos)}-{random.randint(200,999)}-{random.randint(1000,9999)}"

TIPOS_METALES = [
    "Tornillo",
    "Tuerca",
    "Arandela",
    "Clavo",
    "Perno",
    "Broca"
]

TIPOS_SERVICIOS = [
    "Reclamo",
    "Solicitud",
    "Consulta",
    "Cancelación",
    "Cambio de divisa",
    "Pago",
    "Retiro"
]

TELEFONOS = list(set([generar_telefono_do() for _ in range(2000)]))
CODIGOS_PRODUCTOS = list(set([fake.bothify(text=f'{random.choice(TIPOS_METALES)} ????-########', letters='ABCDE') for _ in range(12000)]))
CODIGOS_ORDENES = list(set(["Compra #" + fake.bothify(text='??###', letters='NFXAHTUES') for _ in range(200)]))
CODIGOS_SERVICIOS = list(set([fake.bothify(text=f'{random.choice(TIPOS_SERVICIOS)} ??-###', letters='PQEJ') for _ in range(60)]))

PRODUCT_BRANDS = [
    "AMD Ryzen # ####X",
    "Intel Core i#-###K",
    "NVIDIA GeForce ###",
    "NVIDIA RTX ####",
    "Dell ?###",
    "HP Gen # ??-?###",
    "MacBook Pro ##",
    "MacBook Air ##"
]

PRODUCT_BRANDS = list(set([fake.bothify(text=random.choice(PRODUCT_BRANDS)) for _ in range(200)]))

OPCIONES_PIZZA = [
    "Margarita",
    "Pepperoni",
    "Hawaiana",
    "Cuatro quesos",
    "Vegetariana",
    "Napolitana",
    "Barbacoa",
    "Mexicana",
    "Caprese",
    "Prosciutto y rúcula",
    "Funghi (champiñones)",
    "Atún",
    "Pollo al pesto",
    "Carbonara",
    "Marinara",
    "Chicago style",
    "Siciliana",
    "Mediterránea",
    "Picante",
    "Bianca"
]
PIZZAS = list(set([random.choice(OPCIONES_PIZZA) + " #" + fake.bothify(text='??##') for _ in range(200)]))

OPCIONES_POSTRE = [
    "Pastel de chocolate",
    "Tarta de manzana",
    "Cupcakes",
    "Brownies",
    "Galletas",
    "Cheesecake",
    "Éclairs",
    "Macarons",
    "Panetone",
    "Donas"
]
POSTRES = list(set([random.choice(OPCIONES_POSTRE) + " #" + fake.bothify(text='??##') for _ in range(200)]))
BOOKS = list(set([en_fake.sentence(nb_words=3) +
                  random.choice([" Vol. " + random.choice(["1", "2", "3"]), "", "", " Limited Edition"]) for _ in range(2000)]))
OPCIONES_QUESO = [
    "Cheddar",
    "Mozzarella",
    "Parmesan",
    "Brie",
    "Camembert",
    "Gorgonzola",
    "Roquefort",
    "Gouda",
    "Edam",
    "Swiss",
    "Monterey Jack",
    "Provolone",
    "Feta",
    "Ricotta",
    "Havarti",
    "Gruyere",
    "Blue Stilton",
    "Manchego",
    "Pecorino",
    "Halloumi"
]

plastic_blowed_objects = [
    "Botella de agua",
    "Botella de refresco",
    "Envase de leche",
    "Envase de jugo",
    "Botella de aceite",
    "Frasco de detergente",
    "Recipiente para cosméticos",
    "Biberón",
    "Envase de shampoo",
    "Botella de vino plástico",
    "Bidón de gasolina",
    "Tubo de plástico hueco",
    "Botella de condimento",
    "Botella de cerveza plástico",
    "Contenedor de alimentos",
    "Envase para yogurt",
    "Botella de agua deportiva",
    "Dispenser de líquido",
]

cristaleria = [
    "Copa de vino tinto",
    "Copa de vino blanco",
    "Copa de champán",
    "Copa de cóctel",
    "Copa de licor",
    "Vaso de whisky",
    "Vaso de agua",
    "Vaso de jugo",
    "Vaso alto (highball)",
    "Vaso bajo (old fashioned)",
    "Copa de martini",
    "Copa de margarita",
    "Copa de brandy",
    "Copa de cerveza",
    "Copa de postre",
    "Copa de cordial",
    "Copa de vino rosado",
    "Vaso de cerveza tipo pinta",
    "Vaso de shot",
    "Jarra de vidrio"
]

porcelana = [
    "Plato llano",
    "Plato hondo",
    "Plato de postre",
    "Plato de pan",
    "Plato de servir",
    "Taza de café",
    "Taza de té",
    "Taza con platillo",
    "Tazón pequeño",
    "Tazón grande",
    "Sopera",
    "Cazuela de porcelana",
    "Plato para sopa",
    "Plato para ensalada",
    "Plato de desayuno",
    "Vasija decorativa",
    "Jarra de porcelana",
    "Taza espresso",
    "Plato de frutas",
    "Plato de aperitivo"
]

tipos_de_carros = [
    "Carro",
    "Yipeta",
    "Camioneta",
    "Camión",
    "Bus",
    "Microbús",
    "Motocicleta",
    "Bicicleta",
    "Furgoneta",
    "Pick-up",
    "SUV",
    "Minivan",
    "Tráiler",
    "Autobús articulado",
    "Camión cisterna",
    "Camión de carga",
    "Tractor",
    "Vehículo todoterreno",
    "Vehículo deportivo",
    "Limusina"
]

construcciones = [
    "Apartamento",
    "Casa",
    "Edificio comercial",
    "Oficina",
    "Bodega",
    "Fábrica",
    "Tienda",
    "Escuela",
    "Hospital",
    "Hotel"
]


QUESOS = list(set([random.choice(OPCIONES_QUESO) + " #" + fake.bothify(text='??##') for _ in range(1000)]))
PLASTICOS = list(set([random.choice(plastic_blowed_objects) + " #" + fake.bothify(text='??##') for _ in range(1000)]))
CRISTALES = list(set([random.choice(cristaleria) + " #" + fake.bothify(text='??##') for _ in range(2000)]))
PORCELANAS = list(set([random.choice(porcelana) + " #" + fake.bothify(text='??##') for _ in range(2000)]))
CONSTRUCCIONES = list(set([random.choice(construcciones) + " #" + fake.bothify(text='??##') for _ in range(2000)]))
CARROS = list(set([random.choice(tipos_de_carros) + " #" + fake.bothify(text='??##') for _ in range(20000)]))

tipos_ropa = [
    "Camiseta",
    "Pantalón",
    "Camisa",
    "Falda",
    "Vestido",
    "Chaqueta",
    "Abrigo",
    "Suéter",
    "Shorts",
    "Blusa",
    "Jeans",
    "Traje de baño",
    "Pijama",
    "Sudadera",
    "Chaleco",
    "Traje formal",
    "Bufanda",
    "Guantes",
    "Medias",
    "Gorra"
]
ROPA = list(set([random.choice(tipos_ropa) + " #" + fake.bothify(text='??##') for _ in range(1000)]))

In [14]:
# JOB TO MACHINE PROBLEMSET

from tqdm import tqdm

job_to_machine_configs = [
    # n_jobs, n_machines, min_time, max_time, distributions, friendly_name, job_names, machine_names
    [10, 3, 1, 10, "all", "delivery", random.sample(DIRECCIONES, 10), random.sample(NOMBRES_MASCULINOS, 3)],
    [30, 4, 5, 60, "all", "banco", random.sample(CODIGOS_SERVICIOS, 30), random.sample(NOMBRES, 4)],
    [150, 10, 30, 300, "all", "reparacion", random.sample(PRODUCT_BRANDS, 150), random.sample(NOMBRES, 10)],
    [150, 20, 5, 20, "all", "supermercado", random.sample(CODIGOS_ORDENES, 150), random.sample(NOMBRES, 20)],
    [1000, 100, 2, 30, "all", "call_center", random.sample(TELEFONOS, 1000), random.sample(NOMBRES, 100)],
    [10000, 50, 1, 1000, "all", "metalurgia", random.sample(CODIGOS_PRODUCTOS, 10000), [f"Máquina {i}" for i in range(50)]],
]

PREFIX = "Job To Machine"

problemset = []

for config in tqdm(job_to_machine_configs):

    (
        n_jobs, n_machines, min_time, max_time, distributions,
        friendly_name, job_names, machine_names
    ) = config

    if distributions == "all": distributions = ALL_DISTRIBUTIONS

    for distribution in distributions:

        jm_times = JobSchedulingProblem.create_random_jm_times(
            n_jobs=n_jobs, n_machines=n_machines, min_time=min_time, max_time=max_time, distribution=distribution,
            index=job_names, columns=machine_names)

        for capacity_type in ["limited", "normal"]:
            
            if capacity_type == "limited":
                capacity = int(jm_times.min(axis=1).sum()/n_machines)
            else:
                capacity = int(jm_times.min(axis=1).sum()/(n_machines-1))

            problem = JobToMachineProblem(jm_times, capacity=capacity)

            mlf_solution = MLFSolver.solve(problem)
            mlf_metrics = JobSchedulingProblem.compute_metrics(mlf_solution)
            sjf_solution = SJFSolver.solve(problem)
            sjf_metrics = JobSchedulingProblem.compute_metrics(mlf_solution)
            ljf_solution = LJFSolver.solve(problem)
            ljf_metrics = JobSchedulingProblem.compute_metrics(mlf_solution)

            problem_name = f"{friendly_name}-{n_jobs}j-{n_machines}m-{min_time}_{max_time}tr-{distribution}d"
            mlf_solution_path = f"{PREFIX}/problemset/solution/mlf/{problem_name}-{capacity}c_mlf.xlsx"
            sjf_solution_path = f"{PREFIX}/problemset/solution/sjf/{problem_name}-{capacity}c_sjf.xlsx"
            ljf_solution_path = f"{PREFIX}/problemset/solution/ljf/{problem_name}-{capacity}c_ljf.xlsx"
            job_times_path = f"{PREFIX}/problemset/in/{problem_name}.parquet"

            jm_times.to_parquet(job_times_path)
            mlf_solution.to_excel(mlf_solution_path, index=False)
            sjf_solution.to_excel(sjf_solution_path, index=False)
            ljf_solution.to_excel(ljf_solution_path, index=False)
            
            problemset.append({

                "problem_name": problem_name,
                "job_times": job_times_path,
                "capacity": capacity,

                "mlf_solution": mlf_solution_path,
                "mlf_makespan": mlf_metrics["makespan"],
                "mlf_avg_flow": mlf_metrics["avg_flow_time"],
                "mlf_operations_scheduled": mlf_metrics["n_operations_scheduled"],

                "sjf_solution": sjf_solution_path,
                "sjf_makespan": sjf_metrics["makespan"],
                "sjf_avg_flow": sjf_metrics["avg_flow_time"],
                "sjf_operations_scheduled": sjf_metrics["n_operations_scheduled"],

                "ljf_solution": ljf_solution_path,
                "ljf_makespan": ljf_metrics["makespan"],
                "ljf_avg_flow": ljf_metrics["avg_flow_time"],
                "ljf_operations_scheduled": ljf_metrics["n_operations_scheduled"],

            })

problemset = pd.DataFrame(problemset)
problemset.to_excel(f"{PREFIX}/problemset/problemset.xlsx", index=False)
problemset

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [03:40<00:00, 36.81s/it]


,problem_name,job_times,capacity,mlf_solution,mlf_makespan,mlf_avg_flow,mlf_operations_scheduled,sjf_solution,sjf_makespan,sjf_avg_flow,sjf_operations_scheduled,ljf_solution,ljf_makespan,ljf_avg_flow,ljf_operations_scheduled
0,delivery-10j-3m-1_10tr-uniformd,Job To Machine/problemset/in/delivery-10j-3m-1...,8,Job To Machine/problemset/solution/mlf/deliver...,8,2,9,Job To Machine/problemset/solution/sjf/deliver...,8,2,9,Job To Machine/problemset/solution/ljf/deliver...,8,2,9
1,delivery-10j-3m-1_10tr-uniformd,Job To Machine/problemset/in/delivery-10j-3m-1...,12,Job To Machine/problemset/solution/mlf/deliver...,11,2,10,Job To Machine/problemset/solution/sjf/deliver...,11,2,10,Job To Machine/problemset/solution/ljf/deliver...,11,2,10
2,delivery-10j-3m-1_10tr-constantd,Job To Machine/problemset/in/delivery-10j-3m-1...,3,Job To Machine/problemset/solution/mlf/deliver...,3,1,9,Job To Machine/problemset/solution/sjf/deliver...,3,1,9,Job To Machine/problemset/solution/ljf/deliver...,3,1,9
3,delivery-10j-3m-1_10tr-constantd,Job To Machine/problemset/in/delivery-10j-3m-1...,5,Job To Machine/problemset/solution/mlf/deliver...,5,1,10,Job To Machine/problemset/solution/sjf/deliver...,5,1,10,Job To Machine/problemset/solution/ljf/deliver...,5,1,10
4,delivery-10j-3m-1_10tr-normald,Job To Machine/problemset/in/delivery-10j-3m-1...,13,Job To Machine/problemset/solution/mlf/deliver...,13,4,8,Job To Machine/problemset/solution/sjf/deliver...,13,4,8,Job To Machine/problemset/solution/ljf/deliver...,13,4,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,metalurgia-10000j-50m-1_1000tr-lognormald,Job To Machine/problemset/in/metalurgia-10000j...,20267,Job To Machine/problemset/solution/mlf/metalur...,20266,100,10000,Job To Machine/problemset/solution/sjf/metalur...,20266,100,10000,Job To Machine/problemset/solution/ljf/metalur...,20266,100,10000
80,metalurgia-10000j-50m-1_1000tr-same_per_machined,Job To Machine/problemset/in/metalurgia-10000j...,2400,Job To Machine/problemset/solution/mlf/metalur...,2400,142,748,Job To Machine/problemset/solution/sjf/metalur...,2400,142,748,Job To Machine/problemset/solution/ljf/metalur...,2400,142,748
81,metalurgia-10000j-50m-1_1000tr-same_per_machined,Job To Machine/problemset/in/metalurgia-10000j...,2448,Job To Machine/problemset/solution/mlf/metalur...,2448,141,763,Job To Machine/problemset/solution/sjf/metalur...,2448,141,763,Job To Machine/problemset/solution/ljf/metalur...,2448,141,763
82,metalurgia-10000j-50m-1_1000tr-same_per_jobd,Job To Machine/problemset/in/metalurgia-10000j...,98847,Job To Machine/problemset/solution/mlf/metalur...,98847,494,9999,Job To Machine/problemset/solution/sjf/metalur...,98847,494,9999,Job To Machine/problemset/solution/ljf/metalur...,98847,494,9999


In [15]:
# FLOW SHOP PROBLEMSET

from tqdm import tqdm

flow_shop_configs = [
    # n_jobs, n_machines, min_time, max_time, noise, distributions, friendly_name, job_names, machine_names
    [10, 3, 5, 60, 0.5, "all", "restaurante", [f"Cliente {i}" for i in range(10)], ["Entrada", "Plato fuerte", "Postre"]],
    [50, 4, 5, 60, 0,"all", "pizzeria", random.sample(PIZZAS, 50), ["Amasado", "Untado de salsa", "Agregado de toppings", "Horneado"]],
    [200, 4, 30, 180, 0,"all", "reposteria", random.sample(POSTRES, 200), ["Mezclado", "Horneado", "Enfriamiento", "Decoración"]],
    [200, 4, 30, 180, 0.5,"all", "editorial", random.sample(BOOKS, 200), ["Edición", "Impresión", "Empastado", "Forrado"]],
    [200, 6, 1, 500, 0.25,"all", "queso", random.sample(QUESOS, 200), ["Calentamiento", "Coagulación", "Secado", "Agregado de sal", "Compresión", "Añejamiento"]],
    
    [100, 6, 5, 60, 0,"all", "jugo", [f"Jugo #J-{i}" for i in range(100)], ["Lavado", "Exprimido", "Filtrado", "Pasteurización", "Mezclado", "Embotellado"]],
    [500, 9, 5, 100, 0,"all", "cerveceria", [f"Cerveza #C-{i}" for i in range(500)], ["Malteado", "Molido", "Trituración", "Lauterización", "Hervido", "Enfriado", "Fermentación", "Carbonatación", "Embotellado"]],
    [500, 6, 100, 1000, 0,"all", "plastico", random.sample(PLASTICOS, 500), ["Preparación", "Secado", "Moldeado", "Enfriamiento", "Soplado", "Terminación"]],
    [1000, 8, 5, 100, 0,"all", "cristaleria", random.sample(CRISTALES, 1000), ["Preparación", "Mezclado", "Fusión", "Refinado", "Moldeado", "Recocido", "Corte", "Lavado"]],
    [1000, 8, 5, 100, 0,"all", "porcelana", random.sample(PORCELANAS, 1000), ["Preparación", "Formado", "Secado", "1er horneado", "Esmaltado", "2ndo horneado", "Pintura", "Decoración"]],
    [1000, 18, 1, 1000, 0,"all", "construccion", random.sample(CONSTRUCCIONES, 1000), ["Preparación", "Excavación", "Cimentación",
        "Creación de base", "Creación de columnas", "Creación de muros", "Creación de pisos", "Albañilería", "Techado", "Colocación de ventanas", "Colocación de puertas", "Tubería", "Cableado eléctrico",
        "Revestido", "Cerámica", "Pintura", "Decoración de interiores", "Paisajismo"]],
    [1000, 100, 1, 100, 0.2,"all", "carros", random.sample(CARROS, 1000), [f"Proceso {i}" for i in range(100)]],
    #[10000, 200, 1, 1000, 0,"all", "telefonos", [f"Telefono #T-{i}" for i in range(10000)], [f"Proceso {i}" for i in range(200)]]
]

PREFIX = "Flow Shop"

problemset = []

for config in tqdm(flow_shop_configs):

    (
        n_jobs, n_machines, min_time, max_time, noise, 
        distributions, friendly_name, job_names, machine_names
    ) = config

    if distributions == "all": distributions = ALL_DISTRIBUTIONS

    for distribution in distributions:

        jm_times = JobSchedulingProblem.create_random_jm_times(
            n_jobs=n_jobs, n_machines=n_machines, min_time=min_time, max_time=max_time, distribution=distribution,
            index=job_names, columns=machine_names, noise=noise)

        problem = FlowShopProblem(jm_times)

        def_solution = FSDefaultOrderSolver.solve(problem)
        def_metrics = JobSchedulingProblem.compute_metrics(def_solution)
        ran_solution = FSRandomSolver.solve(problem, max_trys=20)
        ran_metrics = JobSchedulingProblem.compute_metrics(ran_solution)
        if n_jobs < 500:
            neh_solution = NEHSolver.solve(problem)
            neh_metrics = JobSchedulingProblem.compute_metrics(neh_solution)
        else:
            neh_solution = None
            neh_metrics = None

        problem_name = f"{friendly_name}-{n_jobs}j-{n_machines}m-{min_time}_{max_time}tr-{distribution}d"
        def_solution_path = f"{PREFIX}/problemset/solution/def/{problem_name}_def.xlsx"
        ran_solution_path = f"{PREFIX}/problemset/solution/ran/{problem_name}_ran.xlsx"
        neh_solution_path = f"{PREFIX}/problemset/solution/neh/{problem_name}_neh.xlsx"
        job_times_path = f"{PREFIX}/problemset/in/{problem_name}.parquet"

        jm_times.to_parquet(job_times_path)
        def_solution.to_excel(def_solution_path, index=False)
        ran_solution.to_excel(ran_solution_path, index=False)
        if neh_solution is not None: neh_solution.to_excel(neh_solution_path, index=False)
        
        problemset.append({

            "problem_name": problem_name,
            "job_times": job_times_path,

            "def_solution": def_solution_path,
            "def_makespan": def_metrics["makespan"],
            "def_avg_flow": def_metrics["avg_flow_time"],
            "def_operations_scheduled": def_metrics["n_operations_scheduled"],

            "ran_solution": ran_solution_path,
            "ran_makespan": ran_metrics["makespan"],
            "ran_avg_flow": ran_metrics["avg_flow_time"],
            "ran_operations_scheduled": ran_metrics["n_operations_scheduled"],

            "neh_solution": neh_solution_path if neh_solution is not None else None,
            "neh_makespan": neh_metrics["makespan"] if neh_solution is not None else None,
            "neh_avg_flow": neh_metrics["avg_flow_time"] if neh_solution is not None else None,
            "neh_operations_scheduled": neh_metrics["n_operations_scheduled"] if neh_solution is not None else None,

        })

problemset = pd.DataFrame(problemset)
problemset.to_excel(f"{PREFIX}/problemset/problemset.xlsx", index=False)
problemset

100%|██████████| 12/12 [09:02<00:00, 45.21s/it]


,problem_name,job_times,def_solution,def_makespan,def_avg_flow,def_operations_scheduled,ran_solution,ran_makespan,ran_avg_flow,ran_operations_scheduled,neh_solution,neh_makespan,neh_avg_flow,neh_operations_scheduled
0,restaurante-10j-3m-5_60tr-uniformd,Flow Shop/problemset/in/restaurante-10j-3m-5_6...,Flow Shop/problemset/solution/def/restaurante-...,214.0,49,15,Flow Shop/problemset/solution/ran/restaurante-...,193.0,43,15,Flow Shop/problemset/solution/neh/restaurante-...,167.0,46.0,15.0
1,restaurante-10j-3m-5_60tr-constantd,Flow Shop/problemset/in/restaurante-10j-3m-5_6...,Flow Shop/problemset/solution/def/restaurante-...,40.0,11,15,Flow Shop/problemset/solution/ran/restaurante-...,30.0,10,15,Flow Shop/problemset/solution/neh/restaurante-...,30.0,11.0,15.0
2,restaurante-10j-3m-5_60tr-normald,Flow Shop/problemset/in/restaurante-10j-3m-5_6...,Flow Shop/problemset/solution/def/restaurante-...,246.0,65,15,Flow Shop/problemset/solution/ran/restaurante-...,187.0,69,15,Flow Shop/problemset/solution/neh/restaurante-...,171.0,65.0,15.0
3,restaurante-10j-3m-5_60tr-exponentiald,Flow Shop/problemset/in/restaurante-10j-3m-5_6...,Flow Shop/problemset/solution/def/restaurante-...,132.0,24,15,Flow Shop/problemset/solution/ran/restaurante-...,88.0,30,15,Flow Shop/problemset/solution/neh/restaurante-...,77.0,33.0,15.0
4,restaurante-10j-3m-5_60tr-lognormald,Flow Shop/problemset/in/restaurante-10j-3m-5_6...,Flow Shop/problemset/solution/def/restaurante-...,275.0,72,15,Flow Shop/problemset/solution/ran/restaurante-...,211.0,72,15,Flow Shop/problemset/solution/neh/restaurante-...,198.0,73.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,carros-1000j-100m-1_100tr-normald,Flow Shop/problemset/in/carros-1000j-100m-1_10...,Flow Shop/problemset/solution/def/carros-1000j...,59795.0,14348,80000,Flow Shop/problemset/solution/ran/carros-1000j...,59420.0,14038,80000,None,NaN,NaN,NaN
80,carros-1000j-100m-1_100tr-exponentiald,Flow Shop/problemset/in/carros-1000j-100m-1_10...,Flow Shop/problemset/solution/def/carros-1000j...,28693.0,9345,80000,Flow Shop/problemset/solution/ran/carros-1000j...,28479.0,9036,80000,None,NaN,NaN,NaN
81,carros-1000j-100m-1_100tr-lognormald,Flow Shop/problemset/in/carros-1000j-100m-1_10...,Flow Shop/problemset/solution/def/carros-1000j...,30147.0,6959,80000,Flow Shop/problemset/solution/ran/carros-1000j...,29720.0,6754,80000,None,NaN,NaN,NaN
82,carros-1000j-100m-1_100tr-same_per_machined,Flow Shop/problemset/in/carros-1000j-100m-1_10...,Flow Shop/problemset/solution/def/carros-1000j...,87588.0,34571,80000,Flow Shop/problemset/solution/ran/carros-1000j...,85833.0,33832,80000,None,NaN,NaN,NaN


In [76]:
# JOB SHOP PROBLEMSET

from tqdm import tqdm

NOMBRES = list(set([" ".join(fake.name().split()[:2]) for _ in range(2000)]))
PLACE_CODES = list(set([fake.bothify(text='???').upper() for _ in range(200)]))

job_shop_configs = [
    # n_jobs, n_operations, min_time, max_time, noise, distributions, friendly_name, job_names, machine_names
    [10, 3, 5, 60, 0.5, "all", "tutorias", random.sample(NOMBRES, 10), ["Matemáticas", "Biología", "Química"]],
    [20, 8, 5, 60, 0.5, "all", "fotos", random.sample(NOMBRES, 20), ["Perfil", "Escalera", "Cara", "Completa", "Lat-Sup-Der", "Lat-Sup-Izq", "Lat-Inf-Der", "Lat-Inf-Izq"]],
    [50, 4, 5, 60, 0,"all", "pasaporte", random.sample(NOMBRES, 50), ["Entrevista", "Foto", "Acta", "Huellas"]],
    [100, 4, 30, 180, 0,"all", "laboratorio", random.sample(NOMBRES, 100), ["Hemograma", "PCR", "VIH", "Coprológico"]],
    [200, 3, 30, 180, 0.5,"all", "propiedad", random.sample(NOMBRES, 200), ["Visita", "Cotización", "Reserva"]],
    [200, 6, 1, 500, 0.25,"all", "seguros", random.sample(NOMBRES, 200), [f"P{i}" for i in range(6)]],
    [100, 6, 5, 60, 0,"all", "operacion", random.sample(NOMBRES, 100), ["Radiografía", "Electrocardiograma", "Placa", "Hemograma", "Ecografía", "Resonancia"]],
    [500, 6, 5, 100, 0,"all", "pension", random.sample(NOMBRES, 500), [f"P{i}" for i in range(6)]],
    [500, 8, 100, 1000, 0,"all", "textil", random.sample(ROPA, 500), [f"E{i}" for i in range(8)]],
    [1000, 8, 5, 100, 0.1,"all", "galletas", [f"Lote #G{i}" for i in range(1000)], [f"M{i}" for i in range(8)]],
    [1000, 20, 5, 100, 0.1,"all", "delivery", random.sample(NOMBRES, 1000), random.sample(PLACE_CODES, 20)],
    [1000, 100, 1, 100, 0.2,"all", "distribuidora", random.sample(NOMBRES, 1000), random.sample(PLACE_CODES, 100)],
]

PREFIX = "Job Shop"

problemset = []

for config in tqdm(job_shop_configs):

    (
        n_jobs, n_operations, min_time, max_time, noise, 
        distributions, friendly_name, job_names, machine_names
    ) = config

    if distributions == "all": distributions = ALL_DISTRIBUTIONS

    for distribution in distributions:

        job_times, job_machines = JobSchedulingProblem.create_random_jobshop_dataset(
            n_jobs=n_jobs, n_operations=n_operations, min_time=min_time, max_time=max_time, distribution=distribution,
            job_names=job_names, machine_names=machine_names, noise=noise)

        problem = JobShopProblem(job_times, job_machines)

        def_solution = JSDefaultOrderSolver.solve(problem)
        def_metrics = JobSchedulingProblem.compute_metrics(def_solution)
        ran_solution = JSRandomSolver.solve(problem, max_trys=20)
        ran_metrics = JobSchedulingProblem.compute_metrics(ran_solution)
        if n_jobs < 500:
            spt_solution = SPTSolver.solve(problem)
            spt_metrics = JobSchedulingProblem.compute_metrics(spt_solution)
        else:
            spt_solution = None
            spt_metrics = None

        problem_name = f"{friendly_name}-{n_jobs}j-{n_operations}o-{min_time}_{max_time}tr-{distribution}d"
        def_solution_path = f"{PREFIX}/problemset/solution/def/{problem_name}_def.xlsx"
        ran_solution_path = f"{PREFIX}/problemset/solution/ran/{problem_name}_ran.xlsx"
        spt_solution_path = f"{PREFIX}/problemset/solution/spt/{problem_name}_spt.xlsx"
        job_times_path = f"{PREFIX}/problemset/in/{problem_name}_job_times.parquet"
        job_machines_path = f"{PREFIX}/problemset/in/{problem_name}_job_machines.parquet"

        job_times.to_parquet(job_times_path)
        job_machines.to_parquet(job_machines_path)
        def_solution.to_excel(def_solution_path, index=False)
        ran_solution.to_excel(ran_solution_path, index=False)
        if spt_solution is not None: spt_solution.to_excel(spt_solution_path, index=False)
        
        problemset.append({

            "problem_name": problem_name,
            "job_times": job_times_path,
            "job_machines": job_machines_path,

            "def_solution": def_solution_path,
            "def_makespan": def_metrics["makespan"],
            "def_avg_flow": def_metrics["avg_flow_time"],
            "def_operations_scheduled": def_metrics["n_operations_scheduled"],

            "ran_solution": ran_solution_path,
            "ran_makespan": ran_metrics["makespan"],
            "ran_avg_flow": ran_metrics["avg_flow_time"],
            "ran_operations_scheduled": ran_metrics["n_operations_scheduled"],

            "spt_solution": spt_solution_path if spt_solution is not None else None,
            "spt_makespan": spt_metrics["makespan"] if spt_solution is not None else None,
            "spt_avg_flow": spt_metrics["avg_flow_time"] if spt_solution is not None else None,
            "spt_operations_scheduled": spt_metrics["n_operations_scheduled"] if spt_solution is not None else None,

        })

problemset = pd.DataFrame(problemset)
problemset.to_excel(f"{PREFIX}/problemset/problemset.xlsx", index=False)
problemset

100%|██████████| 12/12 [1:15:01<00:00, 375.13s/it] 


,problem_name,job_times,job_machines,def_solution,def_makespan,def_avg_flow,def_operations_scheduled,ran_solution,ran_makespan,ran_avg_flow,ran_operations_scheduled,spt_solution,spt_makespan,spt_avg_flow,spt_operations_scheduled
0,tutorias-10j-3o-5_60tr-uniformd,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/solution/def/tutorias-10j-...,279,96,15,Job Shop/problemset/solution/ran/tutorias-10j-...,202,87,15,Job Shop/problemset/solution/spt/tutorias-10j-...,379.0,86.0,15.0
1,tutorias-10j-3o-5_60tr-constantd,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/solution/def/tutorias-10j-...,30,11,15,Job Shop/problemset/solution/ran/tutorias-10j-...,30,11,15,Job Shop/problemset/solution/spt/tutorias-10j-...,35.0,9.0,15.0
2,tutorias-10j-3o-5_60tr-normald,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/solution/def/tutorias-10j-...,232,83,15,Job Shop/problemset/solution/ran/tutorias-10j-...,193,75,15,Job Shop/problemset/solution/spt/tutorias-10j-...,339.0,64.0,15.0
3,tutorias-10j-3o-5_60tr-exponentiald,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/solution/def/tutorias-10j-...,94,33,15,Job Shop/problemset/solution/ran/tutorias-10j-...,81,32,15,Job Shop/problemset/solution/spt/tutorias-10j-...,171.0,23.0,15.0
4,tutorias-10j-3o-5_60tr-lognormald,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/in/tutorias-10j-3o-5_60tr-...,Job Shop/problemset/solution/def/tutorias-10j-...,185,70,15,Job Shop/problemset/solution/ran/tutorias-10j-...,138,58,15,Job Shop/problemset/solution/spt/tutorias-10j-...,189.0,51.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,distribuidora-1000j-100o-1_100tr-normald,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/solution/def/distribuidora...,50460,49132,80000,Job Shop/problemset/solution/ran/distribuidora...,50016,48815,80000,None,NaN,NaN,NaN
80,distribuidora-1000j-100o-1_100tr-exponentiald,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/solution/def/distribuidora...,23095,22448,80000,Job Shop/problemset/solution/ran/distribuidora...,22503,21885,80000,None,NaN,NaN,NaN
81,distribuidora-1000j-100o-1_100tr-lognormald,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/solution/def/distribuidora...,27426,26807,80000,Job Shop/problemset/solution/ran/distribuidora...,26727,26030,80000,None,NaN,NaN,NaN
82,distribuidora-1000j-100o-1_100tr-same_per_mach...,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/in/distribuidora-1000j-100...,Job Shop/problemset/solution/def/distribuidora...,51886,50672,80000,Job Shop/problemset/solution/ran/distribuidora...,51187,49917,80000,None,NaN,NaN,NaN
